In [8]:
%%capture
%run ../part_1/part1.ipynb
import math
from collections import Counter
from array import array
from collections import defaultdict

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/u251664/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
field_terms = {
    "title":       list(df["title_terms"]),
    "description": list(df["description_terms"]),
    "category":    list(df["category_terms"]),      # category + sub_category merged ASK WHERE WE HAVE MERCHES THIS
    "brand":       list(df["brand_terms"]),
    "seller":      list(df["seller_terms"]),
    "details":     [build_terms(details_to_text(x)) for x in df["product_details"]],   # values only
}

# how much each field counts in the ranking
weights = {"title": 3, "description": 1, "category": 1, "brand": 2, "seller": 0.5, "details": 2}

pids = df["pid"].tolist()
N = len(df)

Part 1: Indexing 

In [12]:
from collections import defaultdict
def create_index(all_terms):
    """
    Implement the inverted index of ONE field.

    Argument:
    all_terms -- list with one list of terms per document (position in the list = doc_id)

    Returns:
    index -- dictionary term -> [[doc_id, positions_of_the_term_in_that_doc], ...]
    """
    index = defaultdict(list)

    for doc_id, terms in enumerate(all_terms):
        current_doc_index = {}   # { 'term1': [doc_id, [positions]], ... }

        for position, term in enumerate(terms):
            try:
                # the term is already in this document: add the position
                current_doc_index[term][1].append(position)
            except:
                # first time we see the term in this document
                current_doc_index[term] = [doc_id, array('I', [position])]

        # merge the index of this document with the main index
        for term, posting in current_doc_index.items():
            index[term].append(posting)

    return index

In [13]:
# one inverted index per field (hybrid approach)
index = {}
for field in field_terms:
    index[field] = create_index(field_terms[field])
    print(field, "->", len(index[field]), "terms")

title -> 738 terms
description -> 4698 terms
category -> 40 terms
brand -> 402 terms
seller -> 687 terms
details -> 17812 terms


In [14]:
def search(query, top_k=20):
    """
    Return the top_k documents that contain ALL the query terms (in any field), ranked by TF-IDF.
    Output: list of (doc_id, score)
    """
    query_terms = list(set(build_terms(query)))
    if len(query_terms) == 0:
        return []

    # read the postings of the query terms: tf[(field, term)] = {doc_id: times in that field}
    tf = {}
    term_docs = {}                      # term -> documents containing it in any field
    for term in query_terms:
        term_docs[term] = set()
        for field in index:
            tf[(field, term)] = {}
            for doc_id, positions in index[field].get(term, []):
                tf[(field, term)][doc_id] = len(positions)
                term_docs[term].add(doc_id)

    # AND
    docs = None
    for term in query_terms:
        if len(term_docs[term]) == 0:
            return []
        docs = term_docs[term] if docs is None else docs & term_docs[term]

    # TF-IDF score
    results = []
    for doc_id in docs:
        score = 0
        for term in query_terms:
            idf = math.log(N / len(term_docs[term]))
            for field in index:
                if doc_id in tf[(field, term)]:
                    score += weights[field] * (1 + math.log(tf[(field, term)][doc_id])) * idf
        results.append((doc_id, score))

    results.sort(key=lambda x: -x[1])
    return results[:top_k]

def show(query, top_k=5):
    """Print the top_k results of a query."""
    for doc_id, score in search(query, top_k):
        print(round(score, 3), pids[doc_id], df["title"].iat[doc_id], "|", df["brand"].iat[doc_id])

show("women full sleeve sweatshirt cotton")

42.869 SWSFFVK89VKGHBZN Full Sleeve Graphic Print Women Sweatshirt | CupidSto
42.869 SWSFFVK8RYHBBFAH Full Sleeve Graphic Print Women Sweatshirt | CupidSto
42.665 SWSFFVKBCQG5FHPF Full Sleeve Printed Women Sweatshirt | CupidSto
42.665 SWSFFVKBH5YEGFFN Full Sleeve Printed Women Sweatshirt | CupidSto
42.665 SWSFFVKBB8FHEB7R Full Sleeve Printed Women Sweatshirt | CupidSto


In [15]:
# most common terms (to choose the queries)
doc_freq = Counter()
for field in index:
    for term, postings in index[field].items():
        doc_freq[term] += len(postings)       # counts a document once per field it appears in
common = [t for t, c in doc_freq.most_common(40)]
print(common[:40])

['shirt', 'cloth', 'neck', 'cotton', 'accessori', 'wear', 'men', 'women', 'sleev', 'wash', 'solid', 'print', '1', 'round', 'western', 'regular', 'machin', 'topwear', 'blend', 'fit', 'india', 'casual', 'blue', 'black', 'full', 'pack', 'half', 'slim', 'comfort', 'design', 'fabric', 'polo', 'white', 'short', 'multicolor', 'color', 'gentl', 'pure', '2', 'look']


In [16]:
test_queries = {
    1: "men round neck cotton tshirt",
    2: "women printed kurta",
    3: "men slim fit jeans",
    4: "women cotton dress",
    5: "men solid polo tshirt",
}
for qid, q in test_queries.items():
    print(qid, q, "->", len(search(q, top_k=100000)), "documents")

1 men round neck cotton tshirt -> 1102 documents
2 women printed kurta -> 44 documents
3 men slim fit jeans -> 212 documents
4 women cotton dress -> 517 documents
5 men solid polo tshirt -> 95 documents
